In [63]:
# --- Code Block 1: Secure Secret Retrieval (Kaggle Environment) ---
from kaggle_secrets import UserSecretsClient
# 
# **Comment 1: Environment-Specific Import**
# This imports `UserSecretsClient`, which is essential for securely accessing 
# credentials (like the Gemini API key) within the Kaggle platform.
# This ensures that sensitive information is not hardcoded into the notebook, 
# following best security practices for cloud/shared environments.
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("gemini")
# **Comment 3: Secret Retrieval and Naming Convention**
# Fetches the secret value associated with the key named "gemini" 
# (which should contain the actual Gemini API key).
# **Suggestion:** For clarity, consider renaming `secret_value_0` to 
# something more descriptive, like `GEMINI_API_KEY` or `api_key`, 
# especially if multiple secrets are used later.
# 
# **Integration Note for Agent 1:**
# This retrieved key must be passed to the first agent's initialization 
# or configuration function (e.g., within a LangChain/CrewAI configuration) 
# to authenticate its calls to the Gemini model.


In [64]:
# --- Code Block 1: API Key Setup and Environment Configuration ---
import os
# Imports the 'os' module to manage environment variables.
from kaggle_secrets import UserSecretsClient
# Imports the client for secure secret access on Kaggle.

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("gemini")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    # Fetches the "gemini" secret securely from Kaggle.
    # Sets the environment variable expected by the Gemini SDK.
except Exception as e:
    print(
        f"🔑 Authentication Error{e}"
    )
    # Robust error handling for authentication failure.
    # **Best Practice:** Using `try...except` ensures graceful failure if the key is missing.

In [65]:
# --- Code Block 2: Agent Development Kit (ADK) Component Imports ---
from google.adk.agents import Agent, SequentialAgent
# Imports core classes: `Agent` (for individual workers) and 
# `SequentialAgent` (crucial for defining the **three-agent pipeline**).
from google.adk.models.google_llm import Gemini
# Imports the `Gemini` model class, specifying the underlying LLM 
# that the agents will use for their reasoning and generation tasks.
from google.adk.runners import InMemoryRunner
# Imports `InMemoryRunner`, the execution engine for the agent system. 
# Suitable for efficient, local, non-distributed execution of the sequence.
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types
from google.adk.sessions import InMemorySessionService
from google.adk.memory import InMemoryMemoryService
# Imports tools: 
# - `AgentTool`: Enables agents to **call other agents** (useful for complex tasks).
# - `FunctionTool`: For integrating custom Python functions.
# - `Google Search`: Essential for grounding agents with real-time data.



In [66]:
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504], 
)

In [72]:
memory_service = (
    InMemoryMemoryService()
    # Initializes `InMemoryMemoryService`. 
# **Purpose:** This service is used to store and retrieve data across agent turns. 
# In a sequential pipeline, it often holds the **context** (the `output_key` # values like "validate"
) 
session_service=InMemorySessionService()


In [68]:
agent1=Agent(
    name="Agent1",
    # Clear, functional name for logging and tracking.
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
        # Uses the efficient 'flash-lite' model, which is ideal for structured 
        # text-in/text-out tasks like validation and cleaning (cost-effective).
        # Important: Assumes 'retry_config' is defined elsewhere to handle 
        # transient API errors, improving **system resilience**.
    ),
    description="Validates and cleans user topics.",
    # Concise summary of the agent's singular, focused responsibility.
    instruction="""
You are Agent 1: Topic Validator.

Your job:

1. Clean the user’s topic.
   - Fix grammar
   - Make it 3–6 words
   - Remove slang

2. Decide if the topic is researchable.
   A topic is valid only if:
   - It is a clear subject (e.g., “Data Visualization using Python”)
   - It is safe (no hacking/illegal/harmful)
   - It is not too vague (“tell me something”, “help me”)
   - It is not personal info


Strict Rules:
- Do NOT provide research.
- Do NOT rewrite instructions.
- Do NOT add extra text.
Output MUST be a single JSON object with the following keys:
- "cleaned_topic": The 3–6 word cleaned topic OR the original topic if invalid.
- "pipeline_action": "CONTINUE" if valid, or "STOP" if invalid/unsafe.
""",
    # **Core Logic:** The detailed prompt (instruction) provides explicit 
    # cleaning, validation (safety, clarity), and strict output rules.
    # 
    # **Best Practice:** The prompt mandates a specific **JSON output format** # and includes an action key ("pipeline_action"), which is critical for 
    # **seamless transition and control flow** to the next agent.
    # 
    # **Suggestion:** Ensure the prompt clearly states the accepted JSON structure 
    # to minimize parsing errors in subsequent steps.
    tools=[],
    # Correctly has no tools, as its task is internal validation, not external search.
    output_key="validate",
    # Defines the key under which this agent's structured output will be stored 
    # in the runner's context for access by Agent 2.

)




In [69]:
agent2= Agent(
    name="Agent2",
    # Clear functional name
    model=Gemini(
        model="gemini-2.5-flash-lite",
        # Continues using the efficient 'flash-lite' model, which is suitable 
        # for summarizing and structuring data from search results
        retry_options=retry_config
    ),
    description="Researches the cleaned topic.",
    instruction="""
You are Agent 2: Research Agent.

Task:
Given a cleaned topic, perform deep research using web search, websites, documentation, blogs, and trusted sources.

Rules:
1. Only include information from the last 24 months (2 years).
2. Do NOT create a roadmap.
3. Do NOT explain your process.
4. Extract and summarize only factual information.
5. Pass the information only
...
Strict Rules:
- Output MUST be a single JSON object containing all the defined research fields.
- No opinions, no roadmap, no chit chat, no pre-amble.

Definitions:
- key_concepts = fundamental ideas required to understand the topic.
- recent_trends = new developments from the last 2 years.
- important_tools = libraries, frameworks, software used for the topic.
- best_practices = updated, proven methods.
- recommended_resources = recent blogs, docs, courses (with clickable URLs).

Strict Rules:
- No opinions.
- No roadmap.
- No chit chat.
IMPORTANT SAFETY RULE:
- If the input is not valid research JSON from Agent 1, or if the topic is unsafe, harmful, illegal, or rejected by Agent 1, you MUST output exactly:
"INVALID_REQUEST"

Do NOT attempt to be helpful.
Do NOT generate educational content.
Just output: INVALID_REQUEST
""",
    # **Core Logic:** Instructs the agent to perform **deep research** # and **strict filtering** (e.g., "last 24 months").
    # 
    # **Critical Design Point:** The prompt explicitly defines the **five research fields** # (`key_concepts`, `recent_trends`, etc.) which forces the output into a structured, 
    # high-quality format necessary for Agent 3 (Roadmap Generator).
    # 
    # **Security/Flow Control:** Includes a strong safety check (Output: "INVALID_REQUEST") 
    # if input validation fails, ensuring the pipeline terminates safely if Agent 1 
    # detects an issue or fails to pass the correct structure.
    tools=[google_search],
    output_key="resource",
    
    #**Crucial:** Includes the `Google Search` tool, which is necessary for 
    # **grounding the research** and adhering to the 2-year data recency rule.
)

# Initializes a runner. 
# **Integration Note:** Similar to Agent 1, this runner setup is temporary. 
# The final execution will require a `SequentialAgent` pipeline encompassing 
# all three agents and a single run call.

In [70]:
agent3= Agent(
    name="Agent3",
    # Clear functional name.
    model=Gemini(
        model="gemini-2.5-flash-lite",
        # Uses 'flash-lite', which is ideal for this **structured text generation** # task (i.e., transforming JSON into a list format).
        retry_options=retry_config,
        session_service=session_service,
        memory_service=memory_service,
    ),
    description="Researches the cleaned topic.",
    instruction="""
You are Agent 3: Roadmap Builder.

Your task:
Take the research JSON provided by Agent 2 and convert it into a clear, actionable learning roadmap.

Output format:
- A numbered list (15–20 steps)
- Sequential from absolute beginner → advanced
- Each step: short, practical, actionable
- Include tools, milestones, and mini-projects
- Provide important links at the end in a structured format
- Do NOT include unnecessary explanations

Rules:
- Do NOT repeat the research JSON.
- Do NOT restate the prompt.
- Do NOT output anything except the roadmap.
- Keep steps concise but specific.

Structure required:
1. Basics
2. Foundations
3. Hands-on skills
4. Tools & libraries
5. Intermediate concepts
6. Projects
7. Advanced topics
8. Final capstone or specialization

IMPORTANT SAFETY RULE:
- If the input is not valid research JSON from Agent 2, or if the topic is unsafe, harmful, illegal, or rejected by Agent 1, you MUST output exactly:
"INVALID_REQUEST"

Do NOT create a roadmap.
Do NOT attempt to be helpful.
Do NOT generate educational content.
Just output: INVALID_REQUEST

Otherwise (valid research JSON only):
- Produce a 15–20 step roadmap.
- Output ONLY the numbered steps.
- No extra text.

""",
    # **Core Logic:** Instructs the agent to synthesize the research into a 
    # **15–20 step, beginner-to-advanced, actionable roadmap**.
    # 
    # **Critical Constraint:** The prompt enforces a **strict eight-part structure** # (1. Basics, 2. Foundations, ..., 8. Capstone), ensuring the final output 
    # is well-organized and meets the user's "structured roadmap" requirement.
    # 
    # **Safety/Flow Control:** The inclusion of the "INVALID_REQUEST" safety rule 
    # is **necessary for consistency**, ensuring the entire pipeline handles 
    # invalid inputs gracefully and uniformly at the final stage.
    # 
    # **Output Focus:** The strong constraint to output **ONLY the numbered steps** # and "No extra text" is excellent for delivering a clean, API-ready final product.
    tools=[],
    output_key="roadmap",
    # Correctly has no tools, as its task is pure synthesis and restructuring of 
    # internal (Agent 2) data.
    
)

# Initializes a runner. 
# **Integration Note:** The final step must now combine Agent 1, Agent 2, and 
# Agent 3 into a single `SequentialAgent` and execute the pipeline once.


In [71]:
def stop_if_invalid(context):
    # context["current_agent_output"] will contain agent1 output
    output = context["validate"]
    # Defines the function that implements the validation check.
    # context["current_agent_output"] will contain agent1 output
    # via the agent's 'output_key'.

    if isinstance(output, str):
        # Accesses the structured output of Agent 1 (Topic Validator).
        import json
        output = json.loads(output)
        # **Robustness:** Handles cases where the output from the LLM is 
        # returned as a string that needs explicit JSON parsing, preventing 
        # errors if the ADK runner doesn't auto-parse the JSON.

    if "pipeline_action" in output and output["pipeline_action"] == "STOP":
        return True   # Stop sequence
        # **Core Logic:** Checks for the specific "STOP" signal defined in 
        # Agent 1's strict rules. Returning `True` here correctly halts 
        # the subsequent agents.

    return False
    # If the action is "CONTINUE" or the key is missing (defaulting to continue), 
    # the function returns `False`, allowing the pipeline to proceed to Agent 2.


In [ ]:
root_agent = SequentialAgent(
    name="RoadmapMaker",
    # Provides a clear, descriptive name for the entire sequential system.
    sub_agents=[agent1, agent2, agent3],
    early_stopping_condition=stop_if_invalid,  
    session_service=session_service,
    memory_service=memory_service
    # **Core Pipeline Definition:** Specifies the exact order of execution: 
    # 1. Agent 1 (Validator) → 2. Agent 2 (Researcher) → 3. Agent 3 (Builder).
    # This structure is crucial for ensuring Agent 2 receives clean input and 
    # Agent 3 receives structured research.
)

In [ ]:
# Initialize runner
runner = InMemoryRunner(
    agent=root_agent,
    session_service=session_service,
    memory_service=memory_service
)

# Execute pipeline
def generate_roadmap(user_topic: str):
    try:
        result = runner.run(
            agent_name="RoadmapMaker",
            user_message=user_topic,
            session_id="unique_session_id"  # Generate unique IDs in production
        )
        
        # Extract final output
        final_output = result.get("roadmap", "")
        
        if final_output == "INVALID_REQUEST":
            return "❌ Invalid or unsafe topic. Please try a different subject."
        
        return final_output
        
    except Exception as e:
        return f"⚠️ Pipeline error: {str(e)}"

# Example usage
topic = "Data Visualization using Python"
roadmap = generate_roadmap(topic)
print(roadmap)

In [88]:
runner = InMemoryRunner(agent=root_agent)
# Instantiates the final execution runner, linking it to the SequentialAgent.
response = await runner.run_debug(
    "data visualisation using python"
)


 ### Created new session: debug_session_id

User > data visualisation using python
Agent1 > {"cleaned_topic": "Data visualization using Python", "pipeline_action": "CONTINUE"}
Agent2 > ```json
{
  "key_concepts": [
    "Data visualization is the graphical representation of data to reveal patterns, trends, and insights, transforming raw information into visual formats like charts or graphs to make complex data more understandable at a glance.",
    "A data visualization library is a tool that simplifies the process of turning raw data into understandable visual formats, offering customizable options to present information in a visually appealing and comprehensible manner.",
    "Python's ecosystem includes numerous libraries for data visualization, enabling the creation of static, animated, and interactive visualizations for various applications, from simple charts to complex dashboards.",
    "Effective visualization design requires integrating statistical accuracy with perceptual cla

In [84]:

print(response[-1].content)

parts=[Part(
  text="""1.  **Set up your Java Development Kit (JDK) and Integrated Development Environment (IDE).** (Milestone: "Hello, World!" program running)
2.  **Master core Java syntax, data types, and control flow.** (Tool: IDE, JDK; Project: Simple calculator)
3.  **Understand Object-Oriented Programming (OOP) principles: Encapsulation, Inheritance, Polymorphism, Abstraction.** (Milestone: Can explain OOP concepts)
4.  **Explore Java Collections Framework (List, Set, Map, Queue).** (Tool: JavaDocs; Project: Contact list manager)
5.  **Learn Exception Handling (try-catch-finally, custom exceptions).** (Milestone: Can write robust error handling)
6.  **Study File I/O operations (reading and writing files).** (Tool: `java.io` package; Project: Text file analysis tool)
7.  **Dive into Multithreading and Concurrency (`Thread`, `Runnable`, synchronization).** (Milestone: Understand thread safety)
8.  **Explore Lambda Expressions and Functional Interfaces (Java 8+).** (Tool: IDE featu